<a href="https://colab.research.google.com/github/MiguelUTEC26/etl-data-pipeline/blob/main/Aseguradora.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install sqlalchemy psycopg2-binary

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.2/4.2 MB 46.2 MB/s eta 0:00:00


In [ ]:
from sqlalchemy import create_engine

In [48]:
import pandas as pd

In [54]:
url = "https://raw.githubusercontent.com/MiguelUTEC26/etl-data-pipeline/refs/heads/main/data/raw/aseguradoras.csv"
df = pd.read_csv(url)

df.head()


,id_aseguradora,nombre,pais,rating_riesgo
0,1,Aseguradora 1,Costa Rica,Alto
1,2,Aseguradora 2,El Salvador,Bajo
2,3,Aseguradora 3,El Salvador,NaN
3,4,Aseguradora 4,Costa Rica,Medio
4,5,Aseguradora 5,ElSalvador,B


In [55]:
df.shape
df.columns
df.info()
df.isnull().sum()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 15 entries, 0 to 14
Data columns (total 4 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   id_aseguradora  15 non-null     int64 
 1   nombre          15 non-null     object
 2   pais            13 non-null     object
 3   rating_riesgo   12 non-null     object
dtypes: int64(1), object(3)
memory usage: 612.0+ bytes


,0
id_aseguradora,0
nombre,0
pais,2
rating_riesgo,3


In [58]:
aseguradora = df.copy()

aseguradora.columns = aseguradora.columns.str.strip().str.lower()

for col in aseguradora.select_dtypes(include='object').columns:
    aseguradora[col] = aseguradora[col].astype(str).str.strip()

aseguradora = aseguradora.replace(r'^\s*$', pd.NA, regex=True)

aseguradora['pais'] = aseguradora['pais'].str.lower()
aseguradora['nombre'] = aseguradora['nombre'].str.lower()

aseguradora = aseguradora.drop_duplicates()


In [59]:
validos = aseguradora[
    aseguradora['pais'].notna() &
    aseguradora['rating_riesgo'].notna()
].copy()

rechazados = aseguradora[
    aseguradora['pais'].isna() |
    aseguradora['rating_riesgo'].isna()
].copy()


In [63]:
def motivo(row):

    motivos = []

    if pd.isna(row['pais']):
        motivos.append("pais_vacio")

    if pd.isna(row['rating_riesgo']):
        motivos.append("riesgo_vacio")

    return ",".join(motivos)

rechazados["motivo_rechazo"] = rechazados.apply(motivo, axis=1)


In [64]:
validos.to_csv("aseguradora_curated.csv", index=False)

rechazados.to_csv("aseguradora_rejects.csv", index=False)
